Installing necessary libararies

In [ ]:
!apt install g++ libboost-all-dev libssl-dev libgoogle-glog-dev libdouble-conversion-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
g++ is already the newest version (4:11.2.0-1ubuntu1).
g++ set to manually installed.
libboost-all-dev is already the newest version (1.74.0.3ubuntu7).
libssl-dev is already the newest version (3.0.2-0ubuntu1.19).
The following additional packages will be installed:
  libgflags-dev libgflags2.2 libgoogle-glog0v5 libunwind-dev
The following NEW packages will be installed:
  libdouble-conversion-dev libgflags-dev libgflags2.2 libgoogle-glog-dev libgoogle-glog0v5
  libunwind-dev
0 upgraded, 6 newly installed, 0 to remove and 30 not upgraded.
Need to get 2,267 kB of archives.
After this operation, 7,935 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgflags2.2 amd64 2.2.2-2 [78.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libgflags-dev amd64 2.2.2-2 [93.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64

In [ ]:
!apt-get install -qq nvidia-cuda-toolkit
!nvidia-smi


Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package libdebuginfod-common.
(Reading database ... 126338 files and directories currently installed.)
Preparing to unpack .../00-libdebuginfod-common_0.186-1ubuntu0.1_all.deb ...
Unpacking libdebuginfod-common (0.186-1ubuntu0.1) ...
Selecting previously unselected package fonts-dejavu-core.
Preparing to unpack .../01-fonts-dejavu-core_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-core (2.37-2build1) ...
Selecting previously unselected package fonts-dejavu-extra.
Preparing to unpack .../02-fonts-dejavu-extra_2.37-2build1_all.deb ...
Unpacking fonts-dejavu-extra (2.37-2build1) ...
Selecting previously unselected package libcupti11.5:amd64.
Preparing to unpack .../03-libcupti11.5_11.5.114~11.5.1-1ubuntu1_amd64.deb ...
Unpacking libcupti11.5:amd64 (11.5.114~11.5.1-1ubuntu1) ...
Selecting previously unselected package libaccinj64-11.5:amd64.
Preparing to unpack .../04-libaccinj64-11.5_11.

DATA HANDLING

converting tsv file to csr

In [ ]:
'''dataset_link : https://graphchallenge.mit.edu/data-sets/'''

'dataset_link : https://graphchallenge.mit.edu/data-sets/'

download tsv file and upload on colab for testing

In [1]:
import numpy as np
from collections import defaultdict

def convert_to_csr(input_file, vertex_out="vertex.CSR", edge_out="edge.CSR"):
    edges = []
    node_set = set()

    # Read the input file
    with open(input_file, 'r') as f:
        for lineno, line in enumerate(f, 1):
            parts = line.strip().split('\t')  # Using tab separator
            if len(parts) < 2:
                print(f"Skipping malformed line {lineno}: {line.strip()}")
                continue
            try:
                to_node = int(parts[0])
                from_node = int(parts[1])
                # Optional: ignore weight if present
            except ValueError:
                print(f"Skipping line with non-integer values at line {lineno}: {line.strip()}")
                continue

            edges.append((from_node, to_node))
            node_set.update([from_node, to_node])

    # Map node IDs to contiguous 0-based indices
    sorted_nodes = sorted(node_set)
    node_to_idx = {node: idx for idx, node in enumerate(sorted_nodes)}
    V = len(sorted_nodes)

    # Build adjacency list
    adj = defaultdict(list)
    for src, dst in edges:
        u = node_to_idx[src]
        v = node_to_idx[dst]
        adj[u].append(v)

    # Build CSR arrays
    row_offsets = [0]
    col_indices = []
    for i in range(V):
        neighbors = sorted(adj[i])  # Sorted neighbors
        col_indices.extend(neighbors)
        row_offsets.append(len(col_indices))

    # Write binary files (64-bit integers)
    np.array(row_offsets, dtype=np.int64).tofile(vertex_out)
    np.array(col_indices, dtype=np.int64).tofile(edge_out)

    print("✅ CSR conversion complete!")
    print(f"📦 Vertices: {V}")
    print(f"📦 Edges: {len(col_indices)}")
    print(f"📁 Output files: {vertex_out}, {edge_out}")

# Example usage
# convert_to_csr("amazon0302_adj.tsv")


In [33]:
convert_to_csr("roadNet-CA_adj.tsv")
#chnage according to dataset

✅ CSR conversion complete!
📦 Vertices: 1965206
📦 Edges: 5533214
📁 Output files: vertex.CSR, edge.CSR


If importing data from link for a large data_set

In [ ]:
#!wget https://p.gera.io/public/graphs/natural/gsh-2015-tpd.tar.gz

--2025-04-07 06:48:51--  https://p.gera.io/public/graphs/natural/gsh-2015-tpd.tar.gz
Resolving p.gera.io (p.gera.io)... 104.21.112.1, 104.21.32.1, 104.21.64.1, ...
Connecting to p.gera.io (p.gera.io)|104.21.112.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1459243647 (1.4G) [application/octet-stream]
Saving to: ‘gsh-2015-tpd.tar.gz’

gsh-2015-tpd.tar.gz 100%[===================>]   1.36G  32.9MB/s    in 43s     

2025-04-07 06:49:35 (32.1 MB/s) - ‘gsh-2015-tpd.tar.gz’ saved [1459243647/1459243647]



In [ ]:
#!tar -xvf gsh-2015-tpd.tar.gz

./gsh-2015-tpd/
./gsh-2015-tpd/edge.CSR
./gsh-2015-tpd/vertex.CSR


In [ ]:
#!cp gsh-2015-tpd/vertex.CSR .
#!cp gsh-2015-tpd/edge.CSR .


EFG IMPLEMENTATION AND COMPRESSION RATIO

In [34]:
%%writefile csr_compression.cu
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <math.h>
#include <cuda_runtime.h>

// CSR structure
typedef struct {
    uint64_t* vertex_csr;
    uint64_t* edge_csr;
    uint64_t num_vertices;
    uint64_t num_edges;
} CSR;

// EF Layout structure
typedef struct {
    CSR csr;
    uint64_t num_vertices;
    uint64_t num_edges;
    uint64_t* chunk_off_exsum;
    uint64_t offset_size;
    uint64_t data_size;
    uint64_t degree_size;
    uint64_t num_lower_bits_size;
    uint64_t storage_size;
} EFLayout;

// Get unsigned type size (similar to Python implementation)
__host__ __device__ uint8_t get_unsigned_type_size(uint64_t val) {
    if (val < (1ULL << 8)) return 1;
    else if (val < (1ULL << 16)) return 2;
    else if (val < (1ULL << 32)) return 4;
    else return 8;
}

// Helper function to compute bit width
__host__ __device__ int bitWidth(uint64_t value) {
    if (value == 0) return 1;
    return (int)(log2(value) + 1);
}

// Calculate the EF encoded size for a list of neighbors
// Fixed to more accurately match Python implementation
__device__ uint64_t ef_encoded_size(uint64_t* neighbors, uint64_t n) {
    if (n == 0) return 0;

    uint64_t U = neighbors[n-1] + 1;  // max + 1
    double log_ratio = log2(fmax((double)U / n, 2.0));

    // Match Python implementation more closely
    double bits = n * (log_ratio + 2);
    uint64_t bytes = (uint64_t)ceil(bits / 8.0);

    return bytes;
}

// Sort array (simple insertion sort for on-device usage)
__device__ void sort_array(uint64_t* arr, uint64_t n) {
    for (int i = 1; i < n; i++) {
        uint64_t key = arr[i];
        int j = i - 1;
        while (j >= 0 && arr[j] > key) {
            arr[j + 1] = arr[j];
            j--;
        }
        arr[j + 1] = key;
    }
}

// Get neighbors for a vertex
__device__ void get_nbrs(uint64_t vid, uint64_t* vertex_csr, uint64_t* edge_csr,
                          uint64_t* neighbors, uint64_t* count) {
    uint64_t start = vertex_csr[vid];
    uint64_t end = vertex_csr[vid + 1];
    *count = end - start;

    for (uint64_t i = 0; i < *count; i++) {
        neighbors[i] = edge_csr[start + i];
    }
}

// CUDA kernel to compute chunk counts for each vertex
__global__ void compute_chunks_kernel(uint64_t* vertex_csr, uint64_t* edge_csr,
                                       uint64_t num_vertices, uint64_t* num_chunks_vec,
                                       uint64_t max_degree, uint64_t chunk_align_size) {
    uint64_t vid = blockIdx.x * blockDim.x + threadIdx.x;

    if (vid < num_vertices) {
        uint64_t* neighbors = (uint64_t*)malloc(max_degree * sizeof(uint64_t));
        if (neighbors == NULL) return; // Handle out of memory

        uint64_t count;
        get_nbrs(vid, vertex_csr, edge_csr, neighbors, &count);
        sort_array(neighbors, count);

        uint64_t encoded_size = ef_encoded_size(neighbors, count);
        num_chunks_vec[vid] = (encoded_size + chunk_align_size - 1) / chunk_align_size;

        free(neighbors);
    }
}

// Calculate maximum degree in CSR
uint64_t get_max_degree(CSR csr) {
    uint64_t max_degree = 0;
    for (uint64_t i = 0; i < csr.num_vertices; i++) {
        uint64_t degree = csr.vertex_csr[i+1] - csr.vertex_csr[i];
        if (degree > max_degree) {
            max_degree = degree;
        }
    }
    return max_degree;
}

// CPU version of ef_encoded_size for verification
uint64_t cpu_ef_encoded_size(uint64_t* neighbors, uint64_t n) {
    if (n == 0) return 0;

    uint64_t U = neighbors[n-1] + 1;  // max + 1
    double log_ratio = log2(fmax((double)U / n, 2.0));
    double bits = n * (log_ratio + 2);

    return (uint64_t)ceil(bits / 8.0);
}

// Setup EF layout parameters
EFLayout set_ef_params(CSR csr) {
    EFLayout ef_layout;
    ef_layout.csr = csr;
    ef_layout.num_vertices = csr.num_vertices;
    ef_layout.num_edges = csr.num_edges;

    uint8_t val_type_size = get_unsigned_type_size(csr.num_vertices);
    uint8_t edge_type_size = get_unsigned_type_size(csr.num_edges);
    uint64_t max_degree = get_max_degree(csr);
    uint8_t degree_type_size = get_unsigned_type_size(max_degree);
    uint8_t fwd_type_size = get_unsigned_type_size(max_degree * 2);

    uint64_t chunk_align_size = fwd_type_size;

    // Allocate memory for vectors
    uint64_t* num_chunks_vec = (uint64_t*)malloc(csr.num_vertices * sizeof(uint64_t));
    if (num_chunks_vec == NULL) {
        printf("Memory allocation failed for num_chunks_vec\n");
        exit(1);
    }

    // CPU-only version for debugging/fallback
    if (max_degree > 0) {
        // Set all values to zero
        for (uint64_t i = 0; i < csr.num_vertices; i++) {
            num_chunks_vec[i] = 0;
        }

        // Process each vertex
        for (uint64_t i = 0; i < csr.num_vertices; i++) {
            uint64_t start = csr.vertex_csr[i];
            uint64_t end = csr.vertex_csr[i + 1];
            uint64_t degree = end - start;

            if (degree > 0) {
                // Extract and sort neighbors
                uint64_t* neighbors = (uint64_t*)malloc(degree * sizeof(uint64_t));
                if (neighbors == NULL) {
                    printf("Memory allocation failed for neighbors array\n");
                    exit(1);
                }

                for (uint64_t j = 0; j < degree; j++) {
                    neighbors[j] = csr.edge_csr[start + j];
                }

                // Simple insertion sort
                for (uint64_t j = 1; j < degree; j++) {
                    uint64_t key = neighbors[j];
                    int64_t k = j - 1;
                    while (k >= 0 && neighbors[k] > key) {
                        neighbors[k + 1] = neighbors[k];
                        k--;
                    }
                    neighbors[k + 1] = key;
                }

                // Calculate encoding size
                uint64_t encoded_size = cpu_ef_encoded_size(neighbors, degree);
                num_chunks_vec[i] = (encoded_size + chunk_align_size - 1) / chunk_align_size;

                free(neighbors);
            }
        }
    }

    // Calculate exclusive sum for chunk offsets
    ef_layout.chunk_off_exsum = (uint64_t*)malloc((csr.num_vertices + 1) * sizeof(uint64_t));
    if (ef_layout.chunk_off_exsum == NULL) {
        printf("Memory allocation failed for chunk_off_exsum\n");
        exit(1);
    }

    ef_layout.chunk_off_exsum[0] = 0;
    for (uint64_t i = 0; i < csr.num_vertices; i++) {
        ef_layout.chunk_off_exsum[i+1] = ef_layout.chunk_off_exsum[i] + num_chunks_vec[i];
    }

    uint64_t total_chunks = ef_layout.chunk_off_exsum[csr.num_vertices];
    uint8_t chunk_offset_size = get_unsigned_type_size(total_chunks);

    // Calculate sizes more carefully matching Python implementation
    ef_layout.offset_size = (csr.num_vertices + 1) * chunk_offset_size;
    ef_layout.data_size = total_chunks * chunk_align_size;
    ef_layout.degree_size = degree_type_size * csr.num_vertices;
    ef_layout.num_lower_bits_size = 1 * csr.num_vertices;  // 1 byte per vertex
    ef_layout.storage_size = ef_layout.offset_size + ef_layout.data_size +
                             ef_layout.degree_size + ef_layout.num_lower_bits_size;

    free(num_chunks_vec);
    return ef_layout;
}

// Get optimal CSR size
uint64_t get_optimal_csr_size(EFLayout ef_layout) {
    uint8_t row_offset_size = get_unsigned_type_size(ef_layout.num_edges + 1);
    uint8_t col_ind_size = get_unsigned_type_size(ef_layout.num_vertices);
    return (ef_layout.num_vertices + 1) * row_offset_size + (ef_layout.num_edges * col_ind_size);
}

// Read binary CSR files
CSR read_binary_csr(const char* vertex_file, const char* edge_file) {
    CSR csr;
    FILE* f_vertex = fopen(vertex_file, "rb");
    FILE* f_edge = fopen(edge_file, "rb");

    if (!f_vertex || !f_edge) {
        printf("Error opening CSR files\n");
        exit(1);
    }

    // Get file sizes
    fseek(f_vertex, 0, SEEK_END);
    uint64_t vertex_size = ftell(f_vertex) / sizeof(uint64_t);
    rewind(f_vertex);

    fseek(f_edge, 0, SEEK_END);
    uint64_t edge_size = ftell(f_edge) / sizeof(uint64_t);
    rewind(f_edge);

    // Allocate memory
    csr.vertex_csr = (uint64_t*)malloc(vertex_size * sizeof(uint64_t));
    csr.edge_csr = (uint64_t*)malloc(edge_size * sizeof(uint64_t));

    if (csr.vertex_csr == NULL || csr.edge_csr == NULL) {
        printf("Memory allocation failed for CSR arrays\n");
        exit(1);
    }

    // Read data
    fread(csr.vertex_csr, sizeof(uint64_t), vertex_size, f_vertex);
    fread(csr.edge_csr, sizeof(uint64_t), edge_size, f_edge);

    csr.num_vertices = vertex_size - 1;
    csr.num_edges = edge_size;

    fclose(f_vertex);
    fclose(f_edge);

    return csr;
}

// Verify that binary files exist
bool verify_files(const char* vertex_file, const char* edge_file) {
    FILE* f_vertex = fopen(vertex_file, "rb");
    FILE* f_edge = fopen(edge_file, "rb");

    bool result = (f_vertex != NULL && f_edge != NULL);

    if (f_vertex) fclose(f_vertex);
    if (f_edge) fclose(f_edge);

    return result;
}

// Main function
int main() {
    // Verify files exist
    if (!verify_files("vertex.CSR", "edge.CSR")) {
        printf("Error: CSR files not found. Please ensure vertex.CSR and edge.CSR exist in current directory.\n");
        return 1;
    }

    // Read CSR files
    CSR csr = read_binary_csr("vertex.CSR", "edge.CSR");

    // Calculate EF Layout parameters
    EFLayout ef_layout = set_ef_params(csr);

    // Calculate sizes and compression ratio
    uint64_t optimal_csr_size = get_optimal_csr_size(ef_layout);
    uint64_t compressed_size = ef_layout.storage_size;
    double compression_ratio = (double)optimal_csr_size / compressed_size;

    // Print results
    printf("Vertices: %lu, Edges: %lu\n", csr.num_vertices, csr.num_edges);
    printf("Optimal CSR Size: %lu bytes\n", optimal_csr_size);
    printf("Compressed (Elias–Fano) Size: %lu bytes\n", compressed_size);
    printf("Compression Ratio (CSR / EF): %.4f\n", compression_ratio);

    // Free memory
    free(csr.vertex_csr);
    free(csr.edge_csr);
    free(ef_layout.chunk_off_exsum);

    return 0;
}

Overwriting csr_compression.cu


In [35]:
!nvcc csr_compression.cu -o csr_compression -arch=sm_70
!./csr_compression

csr_compression.cu(39): warning #20013-D: calling a constexpr __host__ function("log2") from a __host__ __device__ function("bitWidth") is not allowed. The experimental flag '--expt-relaxed-constexpr' can be used to allow this.
      return (int)(log2(value) + 1);
                   ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

csr_compression.cu(39): warning #20015-D: calling a constexpr __host__ function from a __host__ __device__ function is not allowed. The experimental flag '--expt-relaxed-constexpr' can be used to allow this.
      return (int)(log2(value) + 1);
      ^

csr_compression.cu(39): warning #20013-D: calling a constexpr __host__ function("__gnu_cxx::__enable_if<std::__is_integer<T1> ::__value, double> ::__type  ::std::log2<unsigned long> (T1)") from a __host__ __device__ function("bitWidth") is not allowed. The experimental flag '--expt-relaxed-constexpr' can be used to allow this.

Vertices: 1965206, Edges: 5533214
Optimal CSR Size

Sample sssp traversal for testing purpose

In [ ]:
'''%%writefile sssp_efg.cu
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <math.h>
#include <cuda_runtime.h>
#include <limits.h>

// CSR structure
typedef struct {
    uint64_t* vertex_csr;
    uint64_t* edge_csr;
    uint64_t num_vertices;
    uint64_t num_edges;
} CSR;

// EF Layout structure
typedef struct {
    CSR csr;
    uint64_t num_vertices;
    uint64_t num_edges;
    uint64_t* chunk_off_exsum;
    uint64_t offset_size;
    uint64_t data_size;
    uint64_t degree_size;
    uint64_t num_lower_bits_size;
    uint64_t storage_size;
} EFLayout;

// Get unsigned type size
__host__ __device__ uint8_t get_unsigned_type_size(uint64_t val) {
    if (val < (1ULL << 8)) return 1;
    else if (val < (1ULL << 16)) return 2;
    else if (val < (1ULL << 32)) return 4;
    else return 8;
}

// Helper function to compute bit width
__host__ __device__ int bitWidth(uint64_t value) {
    if (value == 0) return 1;
    return (int)(log2(value) + 1);
}

// Calculate the EF encoded size for a list of neighbors
__device__ uint64_t ef_encoded_size(uint64_t* neighbors, uint64_t n) {
    if (n == 0) return 0;

    uint64_t U = neighbors[n-1] + 1;  // max + 1
    double log_ratio = log2(fmax((double)U / n, 2.0));

    // Match Python implementation more closely
    double bits = n * (log_ratio + 2);
    uint64_t bytes = (uint64_t)ceil(bits / 8.0);

    return bytes;
}

// Sort array (simple insertion sort for on-device usage)
__device__ void sort_array(uint64_t* arr, uint64_t n) {
    for (int i = 1; i < n; i++) {
        uint64_t key = arr[i];
        int j = i - 1;
        while (j >= 0 && arr[j] > key) {
            arr[j + 1] = arr[j];
            j--;
        }
        arr[j + 1] = key;
    }
}

// Get neighbors for a vertex
__device__ void get_nbrs(uint64_t vid, uint64_t* vertex_csr, uint64_t* edge_csr,
                          uint64_t* neighbors, uint64_t* count) {
    uint64_t start = vertex_csr[vid];
    uint64_t end = vertex_csr[vid + 1];
    *count = end - start;

    for (uint64_t i = 0; i < *count; i++) {
        neighbors[i] = edge_csr[start + i];
    }
}

// CPU version of ef_encoded_size for verification
uint64_t cpu_ef_encoded_size(uint64_t* neighbors, uint64_t n) {
    if (n == 0) return 0;

    uint64_t U = neighbors[n-1] + 1;  // max + 1
    double log_ratio = log2(fmax((double)U / n, 2.0));
    double bits = n * (log_ratio + 2);

    return (uint64_t)ceil(bits / 8.0);
}

// Calculate maximum degree in CSR
uint64_t get_max_degree(CSR csr) {
    uint64_t max_degree = 0;
    for (uint64_t i = 0; i < csr.num_vertices; i++) {
        uint64_t degree = csr.vertex_csr[i+1] - csr.vertex_csr[i];
        if (degree > max_degree) {
            max_degree = degree;
        }
    }
    return max_degree;
}

// Read binary CSR files
CSR read_binary_csr(const char* vertex_file, const char* edge_file) {
    CSR csr;
    FILE* f_vertex = fopen(vertex_file, "rb");
    FILE* f_edge = fopen(edge_file, "rb");

    if (!f_vertex || !f_edge) {
        printf("Error opening CSR files\n");
        exit(1);
    }

    // Get file sizes
    fseek(f_vertex, 0, SEEK_END);
    uint64_t vertex_size = ftell(f_vertex) / sizeof(uint64_t);
    rewind(f_vertex);

    fseek(f_edge, 0, SEEK_END);
    uint64_t edge_size = ftell(f_edge) / sizeof(uint64_t);
    rewind(f_edge);

    // Allocate memory
    csr.vertex_csr = (uint64_t*)malloc(vertex_size * sizeof(uint64_t));
    csr.edge_csr = (uint64_t*)malloc(edge_size * sizeof(uint64_t));

    if (csr.vertex_csr == NULL || csr.edge_csr == NULL) {
        printf("Memory allocation failed for CSR arrays\n");
        exit(1);
    }

    // Read data
    fread(csr.vertex_csr, sizeof(uint64_t), vertex_size, f_vertex);
    fread(csr.edge_csr, sizeof(uint64_t), edge_size, f_edge);

    csr.num_vertices = vertex_size - 1;
    csr.num_edges = edge_size;

    fclose(f_vertex);
    fclose(f_edge);

    return csr;
}

// Setup EF layout parameters
EFLayout set_ef_params(CSR csr) {
    EFLayout ef_layout;
    ef_layout.csr = csr;
    ef_layout.num_vertices = csr.num_vertices;
    ef_layout.num_edges = csr.num_edges;

    uint8_t val_type_size = get_unsigned_type_size(csr.num_vertices);
    uint8_t edge_type_size = get_unsigned_type_size(csr.num_edges);
    uint64_t max_degree = get_max_degree(csr);
    uint8_t degree_type_size = get_unsigned_type_size(max_degree);
    uint8_t fwd_type_size = get_unsigned_type_size(max_degree * 2);

    uint64_t chunk_align_size = fwd_type_size;

    // Allocate memory for vectors
    uint64_t* num_chunks_vec = (uint64_t*)malloc(csr.num_vertices * sizeof(uint64_t));
    if (num_chunks_vec == NULL) {
        printf("Memory allocation failed for num_chunks_vec\n");
        exit(1);
    }

    // CPU-only version for calculating chunks
    if (max_degree > 0) {
        // Set all values to zero
        for (uint64_t i = 0; i < csr.num_vertices; i++) {
            num_chunks_vec[i] = 0;
        }

        // Process each vertex
        for (uint64_t i = 0; i < csr.num_vertices; i++) {
            uint64_t start = csr.vertex_csr[i];
            uint64_t end = csr.vertex_csr[i + 1];
            uint64_t degree = end - start;

            if (degree > 0) {
                // Extract and sort neighbors
                uint64_t* neighbors = (uint64_t*)malloc(degree * sizeof(uint64_t));
                if (neighbors == NULL) {
                    printf("Memory allocation failed for neighbors array\n");
                    exit(1);
                }

                for (uint64_t j = 0; j < degree; j++) {
                    neighbors[j] = csr.edge_csr[start + j];
                }

                // Simple insertion sort
                for (uint64_t j = 1; j < degree; j++) {
                    uint64_t key = neighbors[j];
                    int64_t k = j - 1;
                    while (k >= 0 && neighbors[k] > key) {
                        neighbors[k + 1] = neighbors[k];
                        k--;
                    }
                    neighbors[k + 1] = key;
                }

                // Calculate encoding size
                uint64_t encoded_size = cpu_ef_encoded_size(neighbors, degree);
                num_chunks_vec[i] = (encoded_size + chunk_align_size - 1) / chunk_align_size;

                free(neighbors);
            }
        }
    }

    // Calculate exclusive sum for chunk offsets
    ef_layout.chunk_off_exsum = (uint64_t*)malloc((csr.num_vertices + 1) * sizeof(uint64_t));
    if (ef_layout.chunk_off_exsum == NULL) {
        printf("Memory allocation failed for chunk_off_exsum\n");
        exit(1);
    }

    ef_layout.chunk_off_exsum[0] = 0;
    for (uint64_t i = 0; i < csr.num_vertices; i++) {
        ef_layout.chunk_off_exsum[i+1] = ef_layout.chunk_off_exsum[i] + num_chunks_vec[i];
    }

    uint64_t total_chunks = ef_layout.chunk_off_exsum[csr.num_vertices];
    uint8_t chunk_offset_size = get_unsigned_type_size(total_chunks);

    // Calculate sizes more carefully matching Python implementation
    ef_layout.offset_size = (csr.num_vertices + 1) * chunk_offset_size;
    ef_layout.data_size = total_chunks * chunk_align_size;
    ef_layout.degree_size = degree_type_size * csr.num_vertices;
    ef_layout.num_lower_bits_size = 1 * csr.num_vertices;  // 1 byte per vertex
    ef_layout.storage_size = ef_layout.offset_size + ef_layout.data_size +
                             ef_layout.degree_size + ef_layout.num_lower_bits_size;

    free(num_chunks_vec);
    return ef_layout;
}

// Verify that binary files exist
bool verify_files(const char* vertex_file, const char* edge_file) {
    FILE* f_vertex = fopen(vertex_file, "rb");
    FILE* f_edge = fopen(edge_file, "rb");

    bool result = (f_vertex != NULL && f_edge != NULL);

    if (f_vertex) fclose(f_vertex);
    if (f_edge) fclose(f_edge);

    return result;
}

// Custom atomic minimum operation for unsigned int values
// Used to work around the uint64_t issue
__device__ void atomicMin_uint(unsigned int* address, unsigned int val) {
    unsigned int old = *address;
    unsigned int assumed;
    do {
        assumed = old;
        old = atomicCAS(address, assumed, min(val, assumed));
    } while (assumed != old);
}

// SSSP Kernel using EF layout with uint32_t distance array
__global__ void sssp_kernel(uint64_t* vertex_csr, uint64_t* edge_csr,
                           uint64_t num_vertices, unsigned int* d_distances,
                           bool* d_updated, uint64_t max_degree) {
    uint64_t tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid >= num_vertices) return;

    uint64_t* neighbors = (uint64_t*)malloc(max_degree * sizeof(uint64_t));
    if (neighbors == NULL) return; // Handle out of memory

    uint64_t count;
    get_nbrs(tid, vertex_csr, edge_csr, neighbors, &count);

    // For simplicity, assume edge weight is 1
    unsigned int current_dist = d_distances[tid];

    // If vertex is unreachable, don't process its neighbors
    if (current_dist == UINT_MAX) {
        free(neighbors);
        return;
    }

    for (uint64_t i = 0; i < count; i++) {
        uint64_t neighbor = neighbors[i];

        // Skip neighbors that are out of bounds (safety check)
        if (neighbor >= num_vertices) continue;

        unsigned int new_dist = current_dist + 1; // Edge weight = 1

        // Only update if we found a shorter path and new_dist doesn't overflow
        if (new_dist > current_dist) { // Check for overflow
            atomicMin_uint(&d_distances[neighbor], new_dist);
            *d_updated = true;
        }
    }

    free(neighbors);
}

// SSSP algorithm using EF layout
void compute_sssp(CSR csr, uint64_t source_vertex, unsigned int* distances) {
    uint64_t* d_vertex_csr;
    uint64_t* d_edge_csr;
    unsigned int* d_distances;
    bool* d_updated;
    bool h_updated;

    // Allocate memory on device
    cudaMalloc((void**)&d_vertex_csr, (csr.num_vertices + 1) * sizeof(uint64_t));
    cudaMalloc((void**)&d_edge_csr, csr.num_edges * sizeof(uint64_t));
    cudaMalloc((void**)&d_distances, csr.num_vertices * sizeof(unsigned int));
    cudaMalloc((void**)&d_updated, sizeof(bool));

    // Copy data to device
    cudaMemcpy(d_vertex_csr, csr.vertex_csr, (csr.num_vertices + 1) * sizeof(uint64_t), cudaMemcpyHostToDevice);
    cudaMemcpy(d_edge_csr, csr.edge_csr, csr.num_edges * sizeof(uint64_t), cudaMemcpyHostToDevice);

    // Initialize distances - use unsigned int instead of uint64_t
    unsigned int* h_distances = (unsigned int*)malloc(csr.num_vertices * sizeof(unsigned int));
    for (uint64_t i = 0; i < csr.num_vertices; i++) {
        h_distances[i] = UINT_MAX; // Initialize to infinity
    }
    h_distances[source_vertex] = 12; // Set source distance to 0

    cudaMemcpy(d_distances, h_distances, csr.num_vertices * sizeof(unsigned int), cudaMemcpyHostToDevice);

    // Get max degree for neighbor buffer allocation
    uint64_t max_degree = get_max_degree(csr);

    // Set block and grid dimensions
    int blockSize = 256;
    int gridSize = (csr.num_vertices + blockSize - 1) / blockSize;

    // Run SSSP iterations
    int max_iterations = 100; // Limit iterations to avoid infinite loop
    int iteration = 0;

    printf("Starting SSSP from vertex %lu\n", source_vertex);

    do {
        h_updated = false;
        cudaMemcpy(d_updated, &h_updated, sizeof(bool), cudaMemcpyHostToDevice);

        sssp_kernel<<<gridSize, blockSize>>>(d_vertex_csr, d_edge_csr, csr.num_vertices,
                                          d_distances, d_updated, max_degree);

        // Check for kernel errors
        cudaError_t err = cudaGetLastError();
        if (err != cudaSuccess) {
            printf("CUDA Error: %s\n", cudaGetErrorString(err));
            break;
        }

        cudaMemcpy(&h_updated, d_updated, sizeof(bool), cudaMemcpyDeviceToHost);

        iteration++;

        if (iteration % 10 == 0) {
            printf("Iteration %d completed\n", iteration);
        }

    } while (h_updated && iteration < max_iterations);

    printf("SSSP completed in %d iterations\n", iteration);

    // Copy results back to host
    cudaMemcpy(distances, d_distances, csr.num_vertices * sizeof(unsigned int), cudaMemcpyDeviceToHost);

    // Free device memory
    cudaFree(d_vertex_csr);
    cudaFree(d_edge_csr);
    cudaFree(d_distances);
    cudaFree(d_updated);

    free(h_distances);
}

int main() {
    // Verify files exist
    if (!verify_files("vertex.CSR", "edge.CSR")) {
        printf("Error: CSR files not found. Please ensure vertex.CSR and edge.CSR exist in current directory.\n");
        return 1;
    }

    // Read CSR files
    CSR csr = read_binary_csr("vertex.CSR", "edge.CSR");
    printf("Graph loaded: %lu vertices, %lu edges\n", csr.num_vertices, csr.num_edges);

    // Calculate EF Layout parameters
    EFLayout ef_layout = set_ef_params(csr);
    printf("EF Layout prepared, storage size: %lu bytes\n", ef_layout.storage_size);

    // Choose source vertex
    uint64_t source_vertex = 0; // Start from vertex 0

    // Compute SSSP - using unsigned int for distances
    unsigned int* distances = (unsigned int*)malloc(csr.num_vertices * sizeof(unsigned int));
    compute_sssp(csr, source_vertex, distances);

    // Print results (limit to first 20 vertices)
    printf("\nShortest path distances from vertex %lu:\n", source_vertex);
    printf("Vertex\tDistance\n");
    uint64_t print_limit = csr.num_vertices < 20 ? csr.num_vertices : 20;
    for (uint64_t i = 0; i < print_limit; i++) {
        if (distances[i] == UINT_MAX) {
            printf("%lu\tINF\n", i);
        } else {
            printf("%lu\t%u\n", i, distances[i]);
        }
    }
    printf("... (showing first %lu vertices)\n", print_limit);

    // Free memory
    free(csr.vertex_csr);
    free(csr.edge_csr);
    free(ef_layout.chunk_off_exsum);
    free(distances);

    return 0;
}'''

Overwriting sssp_efg.cu


Sample sssp traversal for testing purpose

In [28]:
'''%%writefile run_sssp_efg.py
import os
import subprocess

# Check if the CSR files exist and print their sizes
print("Checking for CSR files...")
for filename in ["vertex.CSR", "edge.CSR"]:
    if os.path.exists(filename):
        filesize = os.path.getsize(filename)
        print(f"Found {filename}: {filesize} bytes")
    else:
        print(f"Warning: {filename} not found")

# Compile the CUDA code
print("\nCompiling CUDA code...")
compile_result = subprocess.run(["nvcc", "sssp_efg.cu", "-o", "sssp_efg","-gencode=arch=compute_70,code=sm_70"],
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Check if compilation was successful
if compile_result.returncode != 0:
    print("Compilation error:")
    print(compile_result.stderr.decode())
else:
    print("Compilation successful!")

    # Run the program
    try:
        print("\nRunning SSSP algorithm...")
        run_result = subprocess.run(["./sssp_efg"],
                                  stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                                  timeout=120)

        print("\nProgram output:")
        print(run_result.stdout.decode())

        if run_result.stderr:
            print("Errors/warnings:")
            print(run_result.stderr.decode())
    except subprocess.TimeoutExpired:
        print("Program execution timed out. Check for infinite loops or very large input data.")'''

'%%writefile run_sssp_efg.py\nimport os\nimport subprocess\n\n# Check if the CSR files exist and print their sizes\nprint("Checking for CSR files...")\nfor filename in ["vertex.CSR", "edge.CSR"]:\n    if os.path.exists(filename):\n        filesize = os.path.getsize(filename)\n        print(f"Found {filename}: {filesize} bytes")\n    else:\n        print(f"Warning: {filename} not found")\n\n# Compile the CUDA code\nprint("\nCompiling CUDA code...")\ncompile_result = subprocess.run(["nvcc", "sssp_efg.cu", "-o", "sssp_efg","-gencode=arch=compute_70,code=sm_70"],\n                               stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n\n# Check if compilation was successful\nif compile_result.returncode != 0:\n    print("Compilation error:")\n    print(compile_result.stderr.decode())\nelse:\n    print("Compilation successful!")\n\n    # Run the program\n    try:\n        print("\nRunning SSSP algorithm...")\n        run_result = subprocess.run(["./sssp_efg"],\n                     

checking if sssp is running or not 

In [ ]:
#!python run_sssp_efg.py

Checking for CSR files...
Found vertex.CSR: 51800 bytes
Found edge.CSR: 201152 bytes

Compiling CUDA code...
Compilation successful!

Running SSSP algorithm...

Program output:
Graph loaded: 6474 vertices, 25144 edges
EF Layout prepared, storage size: 69170 bytes
Starting SSSP from vertex 0
Iteration 10 completed
Iteration 20 completed
Iteration 30 completed
Iteration 40 completed
Iteration 50 completed
Iteration 60 completed
Iteration 70 completed
Iteration 80 completed
Iteration 90 completed
Iteration 100 completed
SSSP completed in 100 iterations

Shortest path distances from vertex 0:
Vertex	Distance
0	12
1	13
2	13
3	13
4	16
5	INF
6	INF
7	15
8	15
9	15
10	15
11	15
12	15
13	15
14	13
15	15
16	15
17	15
18	15
19	15
... (showing first 20 vertices)



EFG FORMATE CUDA SSSP IMPLEMENTATION

In [41]:
%%writefile efg_cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <time.h>
#include <curand_kernel.h>

// CSR Graph Structure
typedef struct {
    unsigned long long* vertex_csr;
    unsigned long long* edge_csr;
    unsigned int num_vertices;
    unsigned int num_edges;
    unsigned int* graph_degrees;
    unsigned long long** graph_neighbors;
} CSRGraph;

// Elias-Fano Layout Structure
typedef struct {
    CSRGraph* csr;
    unsigned int num_vertices;
    unsigned int num_edges;
    unsigned long long* chunk_off_exsum;
    size_t offset_size;
    size_t data_size;
    size_t degree_size;
    size_t num_lower_bits_size;
    size_t storage_size;
} EFLayout;

// Helper function to get the appropriate unsigned type size
unsigned int get_unsigned_type_size(unsigned long long val) {
    if (val < (1ULL << 8)) return 1;
    else if (val < (1ULL << 16)) return 2;
    else if (val < (1ULL << 32)) return 4;
    else return 8;
}

// Read binary CSR files
void read_binary_csr(const char* vertex_file, const char* edge_file,
                     unsigned long long** vertex_csr, unsigned long long** edge_csr,
                     unsigned int* num_vertices, unsigned int* num_edges) {
    FILE* vf = fopen(vertex_file, "rb");
    if (!vf) {
        printf("Failed to open vertex file: %s\n", vertex_file);
        exit(1);
    }

    // Get file size for vertex_csr
    fseek(vf, 0, SEEK_END);
    size_t vertex_file_size = ftell(vf);
    rewind(vf);

    size_t vertex_count = vertex_file_size / sizeof(unsigned long long);
    *num_vertices = vertex_count - 1;

    // Allocate memory for vertex_csr
    *vertex_csr = (unsigned long long*)malloc(vertex_file_size);
    if (!(*vertex_csr)) {
        printf("Failed to allocate memory for vertex_csr\n");
        fclose(vf);
        exit(1);
    }

    // Read vertex_csr
    if (fread(*vertex_csr, sizeof(unsigned long long), vertex_count, vf) != vertex_count) {
        printf("Failed to read vertex file\n");
        free(*vertex_csr);
        fclose(vf);
        exit(1);
    }
    fclose(vf);

    // Now handle edge file
    FILE* ef = fopen(edge_file, "rb");
    if (!ef) {
        printf("Failed to open edge file: %s\n", edge_file);
        free(*vertex_csr);
        exit(1);
    }

    // Get file size for edge_csr
    fseek(ef, 0, SEEK_END);
    size_t edge_file_size = ftell(ef);
    rewind(ef);

    size_t edge_count = edge_file_size / sizeof(unsigned long long);
    *num_edges = edge_count;

    // Allocate memory for edge_csr
    *edge_csr = (unsigned long long*)malloc(edge_file_size);
    if (!(*edge_csr)) {
        printf("Failed to allocate memory for edge_csr\n");
        free(*vertex_csr);
        fclose(ef);
        exit(1);
    }

    // Read edge_csr
    if (fread(*edge_csr, sizeof(unsigned long long), edge_count, ef) != edge_count) {
        printf("Failed to read edge file\n");
        free(*vertex_csr);
        free(*edge_csr);
        fclose(ef);
        exit(1);
    }
    fclose(ef);
}

// Build graph from CSR
void build_graph(CSRGraph* graph) {
    graph->graph_degrees = (unsigned int*)malloc(graph->num_vertices * sizeof(unsigned int));
    graph->graph_neighbors = (unsigned long long**)malloc(graph->num_vertices * sizeof(unsigned long long*));

    for (unsigned int i = 0; i < graph->num_vertices; i++) {
        unsigned long long start = graph->vertex_csr[i];
        unsigned long long end = graph->vertex_csr[i + 1];
        unsigned int degree = end - start;

        graph->graph_degrees[i] = degree;

        if (degree > 0) {
            graph->graph_neighbors[i] = (unsigned long long*)malloc(degree * sizeof(unsigned long long));
            for (unsigned int j = 0; j < degree; j++) {
                graph->graph_neighbors[i][j] = graph->edge_csr[start + j];
            }
        } else {
            graph->graph_neighbors[i] = NULL;
        }
    }
}

// Calculate EF encoded size for a set of neighbors
unsigned int ef_encoded_size(unsigned long long* neighbors, unsigned int n) {
    if (n == 0) return 0;

    unsigned long long U = neighbors[n-1] + 1;
    double log_ratio = log2(fmax(U / (double)n, 2.0));
    double bits = n * (log_ratio + 2);

    return (unsigned int)ceil(bits / 8);
}

// Set Elias-Fano parameters
void set_ef_params(EFLayout* ef_layout) {
    CSRGraph* csr = ef_layout->csr;

    // Find max degree
    unsigned int max_degree = 0;
    for (unsigned int i = 0; i < csr->num_vertices; i++) {
        if (csr->graph_degrees[i] > max_degree) {
            max_degree = csr->graph_degrees[i];
        }
    }

    unsigned int val_type_size = get_unsigned_type_size(csr->num_vertices);
    unsigned int edge_type_size = get_unsigned_type_size(csr->num_edges);
    unsigned int degree_type_size = get_unsigned_type_size(max_degree);
    unsigned int fwd_type_size = get_unsigned_type_size(max_degree * 2);
    unsigned int chunk_align_size = fwd_type_size;

    // Allocate and calculate chunk offsets
    unsigned int* num_chunks_vec = (unsigned int*)malloc(csr->num_vertices * sizeof(unsigned int));

    for (unsigned int i = 0; i < csr->num_vertices; i++) {
        unsigned int encoded_size = 0;
        if (csr->graph_degrees[i] > 0) {
            // Sort neighbors
            qsort(csr->graph_neighbors[i], csr->graph_degrees[i], sizeof(unsigned long long),
                  [](const void* a, const void* b) {
                      return (*(unsigned long long*)a > *(unsigned long long*)b) ? 1 : -1;
                  });

            encoded_size = ef_encoded_size(csr->graph_neighbors[i], csr->graph_degrees[i]);
        }

        num_chunks_vec[i] = (encoded_size + chunk_align_size - 1) / chunk_align_size;
    }

    // Calculate exclusive sum for chunk offsets
    ef_layout->chunk_off_exsum = (unsigned long long*)malloc((csr->num_vertices + 1) * sizeof(unsigned long long));
    ef_layout->chunk_off_exsum[0] = 0;

    for (unsigned int i = 0; i < csr->num_vertices; i++) {
        ef_layout->chunk_off_exsum[i + 1] = ef_layout->chunk_off_exsum[i] + num_chunks_vec[i];
    }

    unsigned long long total_chunks = ef_layout->chunk_off_exsum[csr->num_vertices];
    unsigned int chunk_offset_size = get_unsigned_type_size(total_chunks);

    // Calculate storage sizes
    ef_layout->offset_size = (csr->num_vertices + 1) * chunk_offset_size;
    ef_layout->data_size = total_chunks * chunk_align_size;
    ef_layout->degree_size = degree_type_size * csr->num_vertices;
    ef_layout->num_lower_bits_size = 1 * csr->num_vertices;

    ef_layout->storage_size = ef_layout->offset_size + ef_layout->data_size +
                             ef_layout->degree_size + ef_layout->num_lower_bits_size;

    free(num_chunks_vec);
}

// Get optimal CSR size
size_t get_optimal_csr_size(EFLayout* ef_layout) {
    unsigned int row_offset_size = get_unsigned_type_size(ef_layout->num_edges + 1);
    unsigned int col_ind_size = get_unsigned_type_size(ef_layout->num_vertices);

    return (ef_layout->num_vertices + 1) * row_offset_size +
           (ef_layout->num_edges * col_ind_size);
}

// CUDA kernel for initializing random states
__global__ void init_random_states(curandState *states, unsigned long long seed, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size) {
        curand_init(seed, idx, 0, &states[idx]);
    }
}

// CUDA BFS kernel
__global__ void sssp_bfs_kernel(unsigned int* d_graph_degrees,
                               unsigned long long** d_graph_neighbors,
                               bool* d_visited, int* d_queue, int* d_queue_size,
                               int* d_next_queue, int* d_next_queue_size,
                               unsigned long long* d_edge_traversals) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int queue_size = *d_queue_size;

    if (tid < queue_size) {
        int u = d_queue[tid];
        unsigned int degree = d_graph_degrees[u];

        // Count edges traversed
        atomicAdd(d_edge_traversals, degree);

        // Process neighbors
        for (unsigned int i = 0; i < degree; i++) {
            unsigned long long v = d_graph_neighbors[u][i];
            if (!d_visited[v]) {
                d_visited[v] = true;
                int next_idx = atomicAdd(d_next_queue_size, 1);
                d_next_queue[next_idx] = v;
            }
        }
    }
}

// Single-source shortest path BFS
unsigned long long sssp_bfs_ef(EFLayout* ef_layout, unsigned int src) {
    CSRGraph* csr = ef_layout->csr;
    unsigned int num_vertices = csr->num_vertices;

    // CPU arrays
    bool* visited = (bool*)calloc(num_vertices, sizeof(bool));
    int* queue = (int*)malloc(num_vertices * sizeof(int));
    int* next_queue = (int*)malloc(num_vertices * sizeof(int));
    int queue_size = 1;
    int next_queue_size = 0;

    // Initialize
    visited[src] = true;
    queue[0] = src;

    unsigned long long edge_traversals = 0;

    // BFS
    while (queue_size > 0) {
        for (int i = 0; i < queue_size; i++) {
            int u = queue[i];
            edge_traversals += csr->graph_degrees[u];

            for (unsigned int j = 0; j < csr->graph_degrees[u]; j++) {
                unsigned long long v = csr->graph_neighbors[u][j];
                if (!visited[v]) {
                    visited[v] = true;
                    next_queue[next_queue_size++] = v;
                }
            }
        }

        // Swap queues
        int* temp = queue;
        queue = next_queue;
        next_queue = temp;
        queue_size = next_queue_size;
        next_queue_size = 0;
    }

    free(visited);
    free(queue);
    free(next_queue);

    return edge_traversals;
}

// Find connected vertices for sampling
void find_connected_vertices(CSRGraph* csr, unsigned int** connected_vertices, unsigned int* num_connected) {
    unsigned int count = 0;
    for (unsigned int i = 0; i < csr->num_vertices; i++) {
        if (csr->graph_degrees[i] > 0) {
            count++;
        }
    }

    *num_connected = count;
    *connected_vertices = (unsigned int*)malloc(count * sizeof(unsigned int));

    count = 0;
    for (unsigned int i = 0; i < csr->num_vertices; i++) {
        if (csr->graph_degrees[i] > 0) {
            (*connected_vertices)[count++] = i;
        }
    }
}

// Average GTEPS calculation
double average_gteps_ef(EFLayout* ef_layout, unsigned int trials) {
    CSRGraph* csr = ef_layout->csr;

    // Find connected vertices
    unsigned int* connected_vertices;
    unsigned int num_connected;
    find_connected_vertices(csr, &connected_vertices, &num_connected);

    if (num_connected < trials) {
        printf("Only %u connected vertices found, reducing trials.\n", num_connected);
        trials = num_connected;
    }

    // Sample random sources
    unsigned int* random_sources = (unsigned int*)malloc(trials * sizeof(unsigned int));

    // Simple random sampling without replacement
    if (trials == num_connected) {
        // Use all connected vertices
        for (unsigned int i = 0; i < trials; i++) {
            random_sources[i] = connected_vertices[i];
        }
    } else {
        // Fisher-Yates shuffle for sampling
        for (unsigned int i = 0; i < trials; i++) {
            unsigned int j = i + rand() % (num_connected - i);
            // Swap
            unsigned int temp = connected_vertices[i];
            connected_vertices[i] = connected_vertices[j];
            connected_vertices[j] = temp;
            random_sources[i] = connected_vertices[i];
        }
    }

    double total_gteps = 0.0;

    for (unsigned int i = 0; i < trials; i++) {
        unsigned int src = random_sources[i];

        clock_t start = clock();
        unsigned long long edges = sssp_bfs_ef(ef_layout, src);
        clock_t end = clock();

        double duration = ((double)(end - start)) / CLOCKS_PER_SEC;
        double gteps = edges / (duration * 1e9);
        total_gteps += gteps;
    }

    free(connected_vertices);
    free(random_sources);

    return total_gteps / trials;
}

// Free CSR Graph resources
void free_csr_graph(CSRGraph* graph) {
    for (unsigned int i = 0; i < graph->num_vertices; i++) {
        if (graph->graph_neighbors[i]) {
            free(graph->graph_neighbors[i]);
        }
    }

    free(graph->graph_neighbors);
    free(graph->graph_degrees);
    free(graph->vertex_csr);
    free(graph->edge_csr);
}

// Free EF Layout resources
void free_ef_layout(EFLayout* ef_layout) {
    free(ef_layout->chunk_off_exsum);
}

int main() {
    // Start overall runtime measurement
    clock_t total_start_time = clock();

    // Read CSR files
    unsigned long long* vertex_csr;
    unsigned long long* edge_csr;
    unsigned int num_vertices, num_edges;

    printf("Starting to read CSR files...\n");
    clock_t read_start = clock();
    read_binary_csr("vertex.CSR", "edge.CSR", &vertex_csr, &edge_csr, &num_vertices, &num_edges);
    clock_t read_end = clock();
    double read_time = ((double)(read_end - read_start)) / CLOCKS_PER_SEC;
    printf("CSR files read in %.3f seconds\n", read_time);

    // Create CSR Graph
    CSRGraph csr_graph = {
        .vertex_csr = vertex_csr,
        .edge_csr = edge_csr,
        .num_vertices = num_vertices,
        .num_edges = num_edges
    };

    // Build graph
    printf("Building graph...\n");
    clock_t build_start = clock();
    build_graph(&csr_graph);
    clock_t build_end = clock();
    double build_time = ((double)(build_end - build_start)) / CLOCKS_PER_SEC;
    printf("Graph built in %.3f seconds\n", build_time);

    // Create EF Layout
    EFLayout ef_layout = {
        .csr = &csr_graph,
        .num_vertices = num_vertices,
        .num_edges = num_edges
    };

    // Set EF parameters
    printf("Setting EF parameters...\n");
    clock_t ef_start = clock();
    set_ef_params(&ef_layout);
    clock_t ef_end = clock();
    double ef_time = ((double)(ef_end - ef_start)) / CLOCKS_PER_SEC;
    printf("EF parameters set in %.3f seconds\n", ef_time);

    // Print stats
    printf("\n--- Graph Statistics ---\n");
    printf("Vertices: %u, Edges: %u\n", csr_graph.num_vertices, csr_graph.num_edges);
    size_t optimal_csr_size = get_optimal_csr_size(&ef_layout);
    printf("Optimal CSR Size: %zu bytes\n", optimal_csr_size);
    printf("Compressed (Elias–Fano) Size: %zu bytes\n", ef_layout.storage_size);
    printf("Compression Ratio (CSR / EF): %.4f\n",
           (double)optimal_csr_size / ef_layout.storage_size);

    // Run GTEPS benchmark
    unsigned int trials = 100;
    printf("\nRunning SSSP benchmark with %u trials...\n", trials);
    clock_t bench_start = clock();
    double avg_gteps = average_gteps_ef(&ef_layout, trials);
    clock_t bench_end = clock();
    double bench_time = ((double)(bench_end - bench_start)) / CLOCKS_PER_SEC;

    printf("\n--- SSSP on EFG Benchmark Results ---\n");
    printf("Average GTEPS over %u connected runs: %.6f\n", trials, avg_gteps);
    printf("Benchmark completed in %.3f seconds\n", bench_time);

    // Clean up
    printf("\nCleaning up resources...\n");
    free_ef_layout(&ef_layout);
    free_csr_graph(&csr_graph);

    // Calculate and print total runtime
    clock_t total_end_time = clock();
    double total_runtime = ((double)(total_end_time - total_start_time)) / CLOCKS_PER_SEC;
    printf("\n--- Program Summary ---\n");
    printf("Total runtime: %.3f seconds\n", total_runtime);
    printf("File reading: %.3f seconds (%.1f%%)\n", read_time, (read_time/total_runtime)*100);
    printf("Graph building: %.3f seconds (%.1f%%)\n", build_time, (build_time/total_runtime)*100);
    printf("EF parameter setup: %.3f seconds (%.1f%%)\n", ef_time, (ef_time/total_runtime)*100);
    printf("SSSP benchmark: %.3f seconds (%.1f%%)\n", bench_time, (bench_time/total_runtime)*100);

    return 0;
}

Overwriting efg_cuda.cu


In [42]:
# Compile the CUDA code
!nvcc -o  efg_cuda efg_cuda.cu

# Make sure the CSR files are accessible
!ls -lh vertex.CSR edge.CSR

# Run the compiled program
!./efg_cuda

-rw-r--r-- 1 root root 43M Apr  8 07:17 edge.CSR
-rw-r--r-- 1 root root 15M Apr  8 07:17 vertex.CSR
Starting to read CSR files...
CSR files read in 0.034 seconds
Building graph...
Graph built in 0.125 seconds
Setting EF parameters...
EF parameters set in 0.201 seconds

--- Graph Statistics ---
Vertices: 1965206, Edges: 5533214
Optimal CSR Size: 29993684 bytes
Compressed (Elias–Fano) Size: 26391212 bytes
Compression Ratio (CSR / EF): 1.1365

Running SSSP benchmark with 100 trials...

--- SSSP on EFG Benchmark Results ---
Average GTEPS over 100 connected runs: 0.030550
Benchmark completed in 17.975 seconds

Cleaning up resources...

--- Program Summary ---
Total runtime: 18.364 seconds
File reading: 0.034 seconds (0.2%)
Graph building: 0.125 seconds (0.7%)
EF parameter setup: 0.201 seconds (1.1%)
SSSP benchmark: 17.975 seconds (97.9%)


SSSP USING CSR TRAVERSAL

In [43]:
%%writefile graph_ef_cuda.cu

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <stdint.h>
#include <string.h>
#include <cuda_runtime.h>

// CSR structure for graph representation
typedef struct {
    uint64_t* vertex_csr;
    uint64_t* edge_csr;
    uint64_t num_vertices;
    uint64_t num_edges;
    uint64_t max_degree;
} CSR;

// EF Layout structure for compression
typedef struct {
    CSR* csr;
    uint64_t num_vertices;
    uint64_t num_edges;
    uint64_t* chunk_off_exsum;
    uint64_t offset_size;
    uint64_t data_size;
    uint64_t degree_size;
    uint64_t num_lower_bits_size;
    uint64_t storage_size;
} EFLayout;

// Helper functions
uint64_t get_unsigned_type_size(uint64_t val) {
    if (val < (1ULL << 8))
        return 1;
    else if (val < (1ULL << 16))
        return 2;
    else if (val < (1ULL << 32))
        return 4;
    else
        return 8;
}

uint64_t ef_encoded_size(uint64_t* neighbors, uint64_t n) {
    if (n == 0)
        return 0;

    uint64_t U = neighbors[n-1] + 1;
    double log_ratio = log2(fmax(U / (double)n, 2.0));
    double bits = n * (log_ratio + 2);
    return (uint64_t)ceil(bits / 8);
}

uint64_t get_max_degree(CSR* csr) {
    uint64_t max_degree = 0;
    for (uint64_t i = 0; i < csr->num_vertices; i++) {
        uint64_t start = csr->vertex_csr[i];
        uint64_t end = csr->vertex_csr[i + 1];
        uint64_t degree = end - start;
        if (degree > max_degree)
            max_degree = degree;
    }
    return max_degree;
}

// Initialize CSR structure
CSR* initialize_csr(uint64_t* vertex_csr, uint64_t* edge_csr, uint64_t num_vertices, uint64_t num_edges) {
    CSR* csr = (CSR*)malloc(sizeof(CSR));
    csr->vertex_csr = vertex_csr;
    csr->edge_csr = edge_csr;
    csr->num_vertices = num_vertices;
    csr->num_edges = num_edges;
    csr->max_degree = get_max_degree(csr);
    return csr;
}

// Get neighbors for a vertex
uint64_t* get_nbrs(CSR* csr, uint64_t vid, uint64_t* count) {
    if (vid >= csr->num_vertices) {
        *count = 0;
        return NULL;
    }

    uint64_t start = csr->vertex_csr[vid];
    uint64_t end = csr->vertex_csr[vid + 1];
    *count = end - start;

    if (*count == 0)
        return NULL;

    return &(csr->edge_csr[start]);
}

// Initialize EF Layout
EFLayout* initialize_ef_layout(CSR* csr) {
    EFLayout* ef = (EFLayout*)malloc(sizeof(EFLayout));
    ef->csr = csr;
    ef->num_vertices = csr->num_vertices;
    ef->num_edges = csr->num_edges;

    uint64_t val_type_size = get_unsigned_type_size(ef->num_vertices);
    uint64_t edge_type_size = get_unsigned_type_size(ef->num_edges);
    uint64_t max_degree = csr->max_degree;
    uint64_t degree_type_size = get_unsigned_type_size(max_degree);
    uint64_t fwd_type_size = get_unsigned_type_size(max_degree * 2);
    uint64_t chunk_align_size = fwd_type_size;

    // Temporary array to store sorted neighbors
    uint64_t* sorted_neighbors = (uint64_t*)malloc(max_degree * sizeof(uint64_t));
    uint64_t* num_chunks_vec = (uint64_t*)malloc(ef->num_vertices * sizeof(uint64_t));

    for (uint64_t i = 0; i < ef->num_vertices; i++) {
        uint64_t count;
        uint64_t* neighbors = get_nbrs(csr, i, &count);

        // Copy and sort neighbors
        if (count > 0) {
            memcpy(sorted_neighbors, neighbors, count * sizeof(uint64_t));
            // Simple insertion sort (for small neighborhoods)
            for (uint64_t j = 1; j < count; j++) {
                uint64_t key = sorted_neighbors[j];
                int64_t k = j - 1;
                while (k >= 0 && sorted_neighbors[k] > key) {
                    sorted_neighbors[k + 1] = sorted_neighbors[k];
                    k--;
                }
                sorted_neighbors[k + 1] = key;
            }
        }

        uint64_t encoded_size = ef_encoded_size(sorted_neighbors, count);
        uint64_t num_chunks = (encoded_size + chunk_align_size - 1) / chunk_align_size;
        num_chunks_vec[i] = num_chunks;
    }

    // Calculate exclusive sum
    ef->chunk_off_exsum = (uint64_t*)malloc((ef->num_vertices + 1) * sizeof(uint64_t));
    ef->chunk_off_exsum[0] = 0;

    for (uint64_t i = 0; i < ef->num_vertices; i++) {
        ef->chunk_off_exsum[i + 1] = ef->chunk_off_exsum[i] + num_chunks_vec[i];
    }

    uint64_t total_chunks = ef->chunk_off_exsum[ef->num_vertices];
    uint64_t chunk_offset_size = get_unsigned_type_size(total_chunks);

    ef->offset_size = (ef->num_vertices + 1) * chunk_offset_size;
    ef->data_size = total_chunks * chunk_align_size;
    ef->degree_size = degree_type_size * ef->num_vertices;
    ef->num_lower_bits_size = 1 * ef->num_vertices;
    ef->storage_size = ef->offset_size + ef->data_size + ef->degree_size + ef->num_lower_bits_size;

    free(sorted_neighbors);
    free(num_chunks_vec);

    return ef;
}

// Get optimal CSR size
uint64_t get_optimal_csr_size(EFLayout* ef) {
    uint64_t row_offset_size = get_unsigned_type_size(ef->num_edges + 1);
    uint64_t col_ind_size = get_unsigned_type_size(ef->num_vertices);
    return (ef->num_vertices + 1) * row_offset_size + (ef->num_edges * col_ind_size);
}

// Read binary CSR files
void read_binary_csr(const char* vertex_file, const char* edge_file, uint64_t** vertex_csr, uint64_t** edge_csr, uint64_t* num_vertices, uint64_t* num_edges) {
    FILE* vf = fopen(vertex_file, "rb");
    if (!vf) {
        fprintf(stderr, "Error opening vertex file: %s\n", vertex_file);
        exit(1);
    }

    // Get file size and calculate number of vertices
    fseek(vf, 0, SEEK_END);
    long vf_size = ftell(vf);
    rewind(vf);
    *num_vertices = vf_size / sizeof(uint64_t) - 1;

    // Allocate and read vertex CSR
    *vertex_csr = (uint64_t*)malloc(vf_size);
    if (fread(*vertex_csr, 1, vf_size, vf) != vf_size) {
        fprintf(stderr, "Error reading vertex file\n");
        fclose(vf);
        exit(1);
    }
    fclose(vf);

    FILE* ef = fopen(edge_file, "rb");
    if (!ef) {
        fprintf(stderr, "Error opening edge file: %s\n", edge_file);
        free(*vertex_csr);
        exit(1);
    }

    // Get file size and calculate number of edges
    fseek(ef, 0, SEEK_END);
    long ef_size = ftell(ef);
    rewind(ef);
    *num_edges = ef_size / sizeof(uint64_t);

    // Allocate and read edge CSR
    *edge_csr = (uint64_t*)malloc(ef_size);
    if (fread(*edge_csr, 1, ef_size, ef) != ef_size) {
        fprintf(stderr, "Error reading edge file\n");
        fclose(ef);
        free(*vertex_csr);
        exit(1);
    }
    fclose(ef);
}

// CUDA kernel for BFS
__global__ void bfs_kernel(uint64_t* vertex_csr, uint64_t* edge_csr, int* distance, bool* visited,
                           bool* frontier, bool* next_frontier, uint64_t num_vertices, unsigned long long* edge_count) {
    uint64_t v = blockIdx.x * blockDim.x + threadIdx.x;

    if (v < num_vertices && frontier[v]) {
        frontier[v] = false;

        uint64_t start = vertex_csr[v];
        uint64_t end = vertex_csr[v + 1];
        uint64_t edge_traversed = end - start;

        // Count traversed edges once for this vertex
        if (edge_traversed > 0) {
            atomicAdd(edge_count, edge_traversed);
        }

        for (uint64_t i = start; i < end; i++) {
            uint64_t neighbor = edge_csr[i];

            if (!visited[neighbor]) {
                visited[neighbor] = true;
                distance[neighbor] = distance[v] + 1;
                next_frontier[neighbor] = true;
            }
        }
    }
}

// BFS on GPU
uint64_t sssp_bfs_gpu(CSR* csr, uint64_t source) {
    uint64_t num_vertices = csr->num_vertices;

    // Check if source vertex is valid and has neighbors
    if (source >= num_vertices) {
        printf("Invalid source vertex %lu (max: %lu)\n", source, num_vertices - 1);
        return 0;
    }

    uint64_t start = csr->vertex_csr[source];
    uint64_t end = csr->vertex_csr[source + 1];
    if (start == end) {
        printf("Source vertex %lu has no outgoing edges\n", source);
        return 0;
    }

    // Host arrays
    int* h_distance = (int*)malloc(num_vertices * sizeof(int));
    bool* h_visited = (bool*)malloc(num_vertices * sizeof(bool));

    memset(h_distance, -1, num_vertices * sizeof(int));
    memset(h_visited, 0, num_vertices * sizeof(bool));

    h_distance[source] = 0;
    h_visited[source] = true;

    // Device arrays
    uint64_t* d_vertex_csr;
    uint64_t* d_edge_csr;
    int* d_distance;
    bool* d_visited;
    bool* d_frontier;
    bool* d_next_frontier;
    unsigned long long* d_edge_count;

    // Allocate memory on GPU
    cudaMalloc(&d_vertex_csr, (num_vertices + 1) * sizeof(uint64_t));
    cudaMalloc(&d_edge_csr, csr->num_edges * sizeof(uint64_t));
    cudaMalloc(&d_distance, num_vertices * sizeof(int));
    cudaMalloc(&d_visited, num_vertices * sizeof(bool));
    cudaMalloc(&d_frontier, num_vertices * sizeof(bool));
    cudaMalloc(&d_next_frontier, num_vertices * sizeof(bool));
    cudaMalloc(&d_edge_count, sizeof(unsigned long long));

    // Copy data to GPU
    cudaMemcpy(d_vertex_csr, csr->vertex_csr, (num_vertices + 1) * sizeof(uint64_t), cudaMemcpyHostToDevice);
    cudaMemcpy(d_edge_csr, csr->edge_csr, csr->num_edges * sizeof(uint64_t), cudaMemcpyHostToDevice);
    cudaMemcpy(d_distance, h_distance, num_vertices * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_visited, h_visited, num_vertices * sizeof(bool), cudaMemcpyHostToDevice);

    // Initialize frontiers
    bool* h_frontier = (bool*)calloc(num_vertices, sizeof(bool));
    h_frontier[source] = true;

    cudaMemcpy(d_frontier, h_frontier, num_vertices * sizeof(bool), cudaMemcpyHostToDevice);
    cudaMemset(d_next_frontier, 0, num_vertices * sizeof(bool));

    // Initialize edge count to zero
    unsigned long long h_edge_count = 0;
    cudaMemcpy(d_edge_count, &h_edge_count, sizeof(unsigned long long), cudaMemcpyHostToDevice);

    // BFS parameters
    int blockSize = 256;
    int numBlocks = (num_vertices + blockSize - 1) / blockSize;

    // Debug source vertex neighbors
    printf("Source %lu has %lu neighbors\n", source, end - start);

    // BFS iterations
    bool has_frontier = true;
    int iterations = 0;
    while (has_frontier && iterations < 100) {  // Add iteration limit to avoid infinite loops
        iterations++;

        bfs_kernel<<<numBlocks, blockSize>>>(d_vertex_csr, d_edge_csr, d_distance, d_visited,
                                            d_frontier, d_next_frontier, num_vertices, d_edge_count);
        cudaDeviceSynchronize();

        // Check CUDA errors
        cudaError_t error = cudaGetLastError();
        if (error != cudaSuccess) {
            printf("CUDA error: %s\n", cudaGetErrorString(error));
            break;
        }

        // Check if next frontier is empty
        cudaMemcpy(h_frontier, d_next_frontier, num_vertices * sizeof(bool), cudaMemcpyDeviceToHost);

        has_frontier = false;
        for (uint64_t i = 0; i < num_vertices; i++) {
            if (h_frontier[i]) {
                has_frontier = true;
                break;
            }
        }

        // Swap frontiers
        bool* temp = d_frontier;
        d_frontier = d_next_frontier;
        d_next_frontier = temp;

        cudaMemset(d_next_frontier, 0, num_vertices * sizeof(bool));
    }

    // Get final edge count
    cudaMemcpy(&h_edge_count, d_edge_count, sizeof(unsigned long long), cudaMemcpyDeviceToHost);
    printf("BFS completed in %d iterations\n", iterations);
    printf("Edge count from BFS: %llu\n", h_edge_count);

    // Validate results (optional)
    int* final_distance = (int*)malloc(num_vertices * sizeof(int));
    cudaMemcpy(final_distance, d_distance, num_vertices * sizeof(int), cudaMemcpyDeviceToHost);

    int reachable = 0;
    for (uint64_t i = 0; i < num_vertices; i++) {
        if (final_distance[i] >= 0) {
            reachable++;
        }
    }
    printf("Reachable nodes: %d of %lu\n", reachable, num_vertices);

    free(final_distance);
    free(h_distance);
    free(h_visited);
    free(h_frontier);

    cudaFree(d_vertex_csr);
    cudaFree(d_edge_csr);
    cudaFree(d_distance);
    cudaFree(d_visited);
    cudaFree(d_frontier);
    cudaFree(d_next_frontier);
    cudaFree(d_edge_count);

    return (uint64_t)h_edge_count;
}

// Generate random indices of connected vertices
void get_connected_vertices(CSR* csr, uint64_t* connected, uint64_t* count) {
    *count = 0;

    for (uint64_t i = 0; i < csr->num_vertices; i++) {
        uint64_t start = csr->vertex_csr[i];
        uint64_t end = csr->vertex_csr[i + 1];

        if (end > start) {
            connected[(*count)++] = i;

            // Debug: print some connected vertices and their degrees
            if (*count <= 5) {
                printf("Debug: Vertex %lu has %lu neighbors\n", i, end - start);
            }
        }
    }

    printf("Found %lu connected vertices out of %lu total vertices\n", *count, csr->num_vertices);
}

// Average GTEPS calculation
void average_gteps(CSR* csr, int trials) {
    uint64_t num_vertices = csr->num_vertices;

    // Find connected vertices
    uint64_t* connected_vertices = (uint64_t*)malloc(num_vertices * sizeof(uint64_t));
    uint64_t connected_count;

    get_connected_vertices(csr, connected_vertices, &connected_count);

    if (connected_count == 0) {
        printf("Error: No connected vertices found in the graph!\n");
        free(connected_vertices);
        return;
    }

    if (connected_count < trials) {
        printf("Only %lu connected vertices found, reducing trials to %lu.\n", connected_count, connected_count);
        trials = connected_count;
    }

    // Print some diagnostics about the graph
    uint64_t total_edges = 0;
    uint64_t max_degree = 0;
    uint64_t vertex_with_max_degree = 0;

    for (uint64_t i = 0; i < num_vertices; i++) {
        uint64_t degree = csr->vertex_csr[i+1] - csr->vertex_csr[i];
        total_edges += degree;

        if (degree > max_degree) {
            max_degree = degree;
            vertex_with_max_degree = i;
        }
    }

    printf("Graph stats: %lu vertices, %lu edges, average degree: %.2f\n",
           num_vertices, total_edges, (double)total_edges / num_vertices);
    printf("Max degree: %lu at vertex %lu\n", max_degree, vertex_with_max_degree);

    // Random sampling of source vertices - use vertices with significant degree
    uint64_t* random_sources = (uint64_t*)malloc(trials * sizeof(uint64_t));
    srand(time(NULL));

    // Find high-degree vertices for testing
    uint64_t min_degree_threshold = 10;  // Only use vertices with at least this many neighbors
    uint64_t high_degree_count = 0;
    uint64_t* high_degree_vertices = (uint64_t*)malloc(connected_count * sizeof(uint64_t));

    for (uint64_t i = 0; i < connected_count; i++) {
        uint64_t v = connected_vertices[i];
        uint64_t degree = csr->vertex_csr[v+1] - csr->vertex_csr[v];

        if (degree >= min_degree_threshold) {
            high_degree_vertices[high_degree_count++] = v;
        }
    }

    if (high_degree_count > 0) {
        printf("Found %lu vertices with degree >= %lu\n", high_degree_count, min_degree_threshold);

        int actual_trials = trials < high_degree_count ? trials : high_degree_count;

        // Choose random high-degree vertices
        for (int i = 0; i < actual_trials; i++) {
            uint64_t idx = rand() % high_degree_count;
            random_sources[i] = high_degree_vertices[idx];
        }
    } else {
        printf("No vertices with degree >= %lu found, using random connected vertices\n", min_degree_threshold);

        // Choose random connected vertices
        for (int i = 0; i < trials; i++) {
            uint64_t random_index = rand() % connected_count;
            random_sources[i] = connected_vertices[random_index];
        }
    }

    double total_gteps = 0.0;
    int successful_runs = 0;

    for (int i = 0; i < trials; i++) {
        uint64_t src = random_sources[i];

        struct timespec start, end;
        clock_gettime(CLOCK_MONOTONIC, &start);

        uint64_t edges = sssp_bfs_gpu(csr, src);

        clock_gettime(CLOCK_MONOTONIC, &end);

        // Skip vertices that result in zero edges traversed
        if (edges == 0) {
            printf("Run %d from vertex %lu: Skipped (0 edges traversed)\n", i+1, src);
            continue;
        }

        double duration = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;

        // Avoid division by zero
        if (duration < 1e-9) {
            printf("Run %d from vertex %lu: Skipped (duration too short)\n", i+1, src);
            continue;
        }

        double gteps = edges / (duration * 1e9);
        total_gteps += gteps;
        successful_runs++;

        printf("Run %d from vertex %lu: %.6f GTEPS (%lu edges in %.6f seconds)\n",
               i+1, src, gteps, edges, duration);
    }

    if (successful_runs > 0) {
        double avg_gteps = total_gteps / successful_runs;
        printf("Average GTEPS over %d successful runs: %.6f\n", successful_runs, avg_gteps);
    } else {
        printf("No successful runs to average.\n");
    }

    free(connected_vertices);
    free(random_sources);
    free(high_degree_vertices);
}

int main() {
    uint64_t *vertex_csr, *edge_csr;
    uint64_t num_vertices, num_edges;

    // Read CSR files
    read_binary_csr("vertex.CSR", "edge.CSR", &vertex_csr, &edge_csr, &num_vertices, &num_edges);

    // Initialize CSR structure
    CSR* csr = initialize_csr(vertex_csr, edge_csr, num_vertices, num_edges);

    // Initialize EF layout for compression analysis
    EFLayout* ef_layout = initialize_ef_layout(csr);

    // Print statistics
    printf("Vertices: %lu, Edges: %lu\n", csr->num_vertices, csr->num_edges);
    printf("Optimal CSR Size: %lu bytes\n", get_optimal_csr_size(ef_layout));
    printf("Compressed (Elias-Fano) Size: %lu bytes\n", ef_layout->storage_size);
    printf("Compression Ratio (CSR / EF): %.4f\n",
           (double)get_optimal_csr_size(ef_layout) / ef_layout->storage_size);

    printf("\n--- SSSP Average GTEPS Benchmark ---\n");
    // Reduce the number of trials to avoid excessive computation
    average_gteps(csr, 100);

    // Free memory
    free(vertex_csr);
    free(edge_csr);
    free(ef_layout->chunk_off_exsum);
    free(ef_layout);
    free(csr);

    return 0;
}

Overwriting graph_ef_cuda.cu


In [44]:
!nvcc -o graph_ef_cuda graph_ef_cuda.cu -arch=sm_70 -lcudart
!./graph_ef_cuda

Vertices: 1965206, Edges: 5533214
Optimal CSR Size: 29993684 bytes
Compressed (Elias-Fano) Size: 26391212 bytes
Compression Ratio (CSR / EF): 1.1365

--- SSSP Average GTEPS Benchmark ---
Debug: Vertex 0 has 3 neighbors
Debug: Vertex 1 has 3 neighbors
Debug: Vertex 2 has 4 neighbors
Debug: Vertex 3 has 4 neighbors
Debug: Vertex 4 has 3 neighbors
Found 1965206 connected vertices out of 1965206 total vertices
Graph stats: 1965206 vertices, 5533214 edges, average degree: 2.82
Max degree: 12 at vertex 1481106
Found 3 vertices with degree >= 10
Source 1449968 has 10 neighbors
BFS completed in 100 iterations
Edge count from BFS: 236415
Reachable nodes: 76570 of 1965206
Run 1 from vertex 1449968: 0.000383 GTEPS (236415 edges in 0.616555 seconds)
Source 1481106 has 12 neighbors
BFS completed in 100 iterations
Edge count from BFS: 248030
Reachable nodes: 80853 of 1965206
Run 2 from vertex 1481106: 0.000514 GTEPS (248030 edges in 0.482995 seconds)
Source 1449968 has 10 neighbors
BFS completed in 